In [1]:
!pip install faiss-cpu sentence-transformers rank-bm25 pymupdf ollama numpy tqdm

In [2]:
import os
import fitz
import faiss
import numpy as np
import ollama
from tqdm import tqdm
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

e:\WorkAI\Works\Projects\Hybrid_RAG_Complete\GlobalEnv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
RERANK_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
OLLAMA_MODEL = "tinyllama"

CHUNK_SIZE = 500
CHUNK_OVERLAP = 100
TOP_K = 5

PDF_FOLDER = "E:\\WorkAI\\Works\\Projects\\Hybrid_RAG_Complete\\data\\docs_to_ingest"

os.makedirs(PDF_FOLDER, exist_ok=True)

In [4]:
def load_pdfs(folder_path):
    documents = []
    for file in os.listdir(folder_path):
        if file.endswith(".pdf"):
            print(f"Loading: {file}")
            doc = fitz.open(os.path.join(folder_path, file))
            text = ""
            for page in doc:
                text += page.get_text()
            documents.append(text)
    return documents

In [5]:
def chunk_text(text, chunk_size=500, overlap=100):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

In [6]:
def chunk_text(text, chunk_size=500, overlap=100):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

In [7]:
print("Loading embedding model...")
embedding_model = SentenceTransformer(EMBEDDING_MODEL)

print("Loading reranker model...")
reranker = CrossEncoder(RERANK_MODEL)

Loading embedding model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 140.30it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading reranker model...


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 161.67it/s, Materializing param=classifier.weight]                                    
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
class VectorStore:
    def __init__(self, dim):
        self.index = faiss.IndexFlatL2(dim)
        self.texts = []

    def add(self, embeddings, texts):
        self.index.add(np.array(embeddings).astype("float32"))
        self.texts.extend(texts)

    def search(self, query_embedding, top_k=5):
        D, I = self.index.search(np.array([query_embedding]).astype("float32"), top_k)
        return [self.texts[i] for i in I[0]]

In [9]:
def build_pipeline(pdf_folder):
    documents = load_pdfs(pdf_folder)

    chunks = []
    for doc in documents:
        chunks.extend(chunk_text(doc, CHUNK_SIZE, CHUNK_OVERLAP))

    print(f"Total chunks: {len(chunks)}")

    print("Generating embeddings...")
    embeddings = embedding_model.encode(chunks, show_progress_bar=True)

    vector_store = VectorStore(len(embeddings[0]))
    vector_store.add(embeddings, chunks)

    print("Building BM25 index...")
    tokenized = [doc.split() for doc in chunks]
    bm25 = BM25Okapi(tokenized)

    return vector_store, bm25, chunks

In [10]:
def retrieve(query, vector_store, bm25, chunks):
    query_embedding = embedding_model.encode([query])[0]

    dense_results = vector_store.search(query_embedding, TOP_K)
    sparse_results = bm25.get_top_n(query.split(), chunks, n=TOP_K)

    combined = list(set(dense_results + sparse_results))

    pairs = [(query, doc) for doc in combined]
    scores = reranker.predict(pairs)

    ranked = sorted(zip(combined, scores), key=lambda x: x[1], reverse=True)

    return [doc for doc, score in ranked[:TOP_K]]

In [11]:
def generate_answer(query, context_docs):
    context = "\n\n".join(context_docs)

    prompt = f"""
Use ONLY the context below to answer.

Context:
{context}

Question:
{query}

Answer:
"""

    response = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[{"role": "user", "content": prompt}]
    )

    return response["message"]["content"]

In [12]:
print("Building Hybrid RAG System...")
vector_store, bm25, chunks = build_pipeline(PDF_FOLDER)

print("System Ready!")

Building Hybrid RAG System...
Loading: AbidR1.pdf
Loading: httpsdoi.org10.69996jcai.2025007.pdf
Loading: s10796-025-10581-7.pdf
Total chunks: 580
Generating embeddings...


Batches: 100%|██████████| 19/19 [01:07<00:00,  3.53s/it]

Building BM25 index...
System Ready!


In [13]:
query = " Human Oversight and Ethical Literacy is Required"

context_docs = retrieve(query, vector_store, bm25, chunks)
answer = generate_answer(query, context_docs)

print("ANSWER:\n")
print(answer)

ANSWER:

Yes, the participants of the AI research review emphasized that human oversight and ethical literacy are required for the effective and responsible usage of AI in educational institutions and to introduce AI in the education system. The need for human control in the training of AI models and the need for a clear understanding of the composition of datasets were acknowledged, with the participants urging the development of proper policies regarding ethical and generative plagiarism. The authors also observed the urgent necessity for developing proper policies regarding the development of AI in educational institutions and introducing literacy in AI in the education system.


In [14]:
while True:
    query = input("\nAsk a question (type 'exit' to stop): ")

    if query.lower() == "exit":
        break

    context_docs = retrieve(query, vector_store, bm25, chunks)
    answer = generate_answer(query, context_docs)

    print("\nAnswer:\n", answer)